<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.05rem; margin-bottom: 0.2rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.2rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

## Module 6: *Class Balancing*
##### Version Number: 4.0
---
### Contents  
> 1. Data Preparation
> 2. Automatic Class Balancing 
> 3. Export File
---
### Notes

**WARNING** this module is computation heavy
- Start with a **baseline model** for comparison.
- Test with multi-classification **tree-based models** (Random Forest, XGBoost) and LGBM.
- Test and evaluate multiple class balancing strategies (No sampling, Oversampling, Undersampling)
- Compare average F1 score among all classes

---
### Inputs
- `X_scaled`,`y_reduced` Reduced or full size Modeling sets

---
### Outputs  

`best_strategy` - dataframe recording best sampling strategy for each method

---
### User Created Dependencies

In [1]:
# Add the parent directory to the system path so "src" can be found
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

# function to display a formatted report on classification metrics
from src.model_utils import gen_report

# function that accepts any model and performs k fold validation on the test set with that model
from src.model_utils import kfold

---
### Third Party Dependencies

In [2]:
# Core Python libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing and modeling
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
from lightgbm import LGBMClassifier

# Metrics
from sklearn.metrics import classification_report
from sklearn.metrics import mean_squared_error, r2_score

# Resampling tools
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

# Style
sns.set(style='whitegrid')
plt.rcParams["figure.dpi"] = 100

---

## 1. Prepare Data for Modeling - Scaling, Splitting, and Resampling

In [3]:
# Load processed feature and label data
X = pd.read_csv('../data/processed/X.csv')
y = pd.read_csv('../data/processed/y.csv').squeeze()

### 1.2 Split Data

In [4]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

---

## 2. Automatic Class Balancing

To address the extreme inbalance in the dataset, multiple strategies are explored.
- **In Method Balancing** is used when applicable 
- **RandomUnderSampler** will remove random members of the majority class (Low severity) until they are balanced. This creates a much smaller dataset to model.
- **SMOTE (Synthetic Minority Over-sampling Technique)** will generate synthetic members of the minority classes. Introduces potential noise by synthetic sampling

### 2.1 Utility functions

A function to perform both in and out of sample testing with kfold crossvalidation and generate a report of metrics.

In [5]:
def class_balancing(X_train, y_train, X_test, y_test, model, sampling_strategy='No_balance'):
    
    if sampling_strategy == 'Undersampling':
        rus = RandomUnderSampler(sampling_strategy='auto', random_state=14)
        X_train, y_train = rus.fit_resample(X_train, y_train)
   
    if sampling_strategy == 'Oversampling':
        smote = SMOTE()
        X_train, y_train = smote.fit_resample(X_train, y_train)
          
    reports = kfold(X_train, y_train, model)    
    Train_metrics = gen_report(reports)

    # Retrain on full training set
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    reports = [classification_report(y_test, y_pred, output_dict=True)]
    Test_metrics = gen_report(reports)

    # Add context columns
    for df in [Train_metrics, Test_metrics]:
        df['Phase'] = 'Train' if df is Train_metrics else 'Test'
        df['Model'] = model.__class__.__name__
        df['Balancing'] = sampling_strategy

    # Combine and reorder
    combined_metrics = pd.concat([Train_metrics, Test_metrics], axis=0)
    combined_metrics = combined_metrics.reset_index().rename(columns={'index': 'Class'})

    return combined_metrics

### 2.2 Define default machine learning models

In [6]:
default_rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
)

In [7]:
default_lgbm_model = LGBMClassifier(
    n_estimators=500,
    num_leaves=31,
    max_depth=-1,
    learning_rate=0.05,
    class_weight='balanced',
    verbose = -1
)

In [8]:
default_xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
) 

 ###  2.3 Compare Sampling Methods

In [9]:
models = {
    'RF': default_rf_model,
    'LGBM': default_lgbm_model,
    'XGB': default_xgb_model
}

sampling_strategies = ['Undersampling','No_balance','Oversampling']

all_results = []
counter = 1

for strategy in sampling_strategies:
    for name, model in models.items():
        print("running", strategy, counter, "of 9...")
        counter = counter + 1
        result = class_balancing(
            X_train, y_train, X_test, y_test,
            model,
            sampling_strategy=strategy
        )
        result['Model_Label'] = name
        all_results.append(result)

# Combine into one DataFrame
all_results_df = pd.concat(all_results, axis=0).reset_index(drop=True)

running Undersampling 1 of 9...
running Undersampling 2 of 9...
running Undersampling 3 of 9...
running No_balance 4 of 9...
running No_balance 5 of 9...
running No_balance 6 of 9...
running Oversampling 7 of 9...
running Oversampling 8 of 9...
running Oversampling 9 of 9...


### 2.4 Format and display results

In [10]:
strategy_summary = (
    all_results_df[all_results_df['Phase'] == 'Test']
    .groupby(['Balancing','Model_Label'])['F1-Score']
    .mean()
    .reset_index()
    .sort_values(by='F1-Score', ascending=False)
)

strategy_summary_pivot = (
    strategy_summary
    .pivot(index='Model_Label', columns='Balancing', values='F1-Score')
    .sort_values(by='No_balance', ascending=False)
)

strategy_summary_pivot.style.highlight_max(axis=1, color='lightgreen')

Balancing,No_balance,Oversampling,Undersampling
Model_Label,,,
LGBM,0.842322,0.843690,0.833445
XGB,0.814412,0.806182,0.801539
RF,0.765609,0.765579,0.762309


### Display Best Strategies

In [11]:
# Find best balancing strategy per model
best_strategy = strategy_summary_pivot.idxmax(axis=1)

# Combine with model names into a new DataFrame
best_strategy_df = pd.DataFrame({
    'Model_Label': best_strategy.index,
    'Best_Strategy': best_strategy.values
})

In [12]:
best_strategy_df

,Model_Label,Best_Strategy
0,LGBM,Oversampling
1,XGB,No_balance
2,RF,No_balance


### Detailed results (ALL RESULTS)

In [13]:
all_results_df

,Class,Category,Precision,Recall,F1-Score,Phase,Model,Balancing,Model_Label
0,0,0,0.809754,0.749721,0.778556,Train,RandomForestClassifier,Undersampling,RF
1,1,1,0.730017,0.702873,0.716143,Train,RandomForestClassifier,Undersampling,RF
2,2,2,0.798315,0.887175,0.840395,Train,RandomForestClassifier,Undersampling,RF
3,0,0,0.823329,0.757311,0.788941,Test,RandomForestClassifier,Undersampling,RF
4,1,1,0.769304,0.700559,0.733324,Test,RandomForestClassifier,Undersampling,RF
5,2,2,0.667367,0.895168,0.764662,Test,RandomForestClassifier,Undersampling,RF
6,0,0,0.830064,0.785880,0.807339,Train,LGBMClassifier,Undersampling,LGBM
7,1,1,0.789331,0.779421,0.784317,Train,LGBMClassifier,Undersampling,LGBM
8,2,2,0.902620,0.962007,0.931356,Train,LGBMClassifier,Undersampling,LGBM
9,0,0,0.835414,0.790626,0.812404,Test,LGBMClassifier,Undersampling,LGBM


---

## 3. Export File

In [14]:
best_strategy_df.to_csv('../data/processed/best_strategy.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
